# CS 467 Humidity Sensor

Filtering and LED pattern conversion prototyping

Algorithm prototypes are implemented in Python, but production algorithms will be implemented in Rust

## Requirements

- Install Python 3.13
- Install `uv`: https://docs.astral.sh/uv/

## Usage

- Install dependencies with `uv` by running `uv sync`
  - This will create a virtual environment in the `.venv` directory with all of the necessary dependencies installed
- Open the notebook file and select the interpreter in `.venv`

## Imports

In [90]:
import pandas as pd
import numpy as np
import plotly.express as px
import random
import ipytest
ipytest.autoconfig()

## Generate Sample Data

In [91]:
random.seed(42)

data = []
for i in range(1, 100):
    data.extend([i] * random.randint(10, 200))

df = pd.DataFrame(data, columns=["Clean Data"])
df.index.name = "Time"

### Add Noise

In [92]:
mu = 0
sigma = 0.5
noise = np.random.normal(mu, sigma, size=len(data)).round(1)

df["Noisy Data"] = df["Clean Data"] + noise

In [93]:
display(df)

,Clean Data,Noisy Data
Time,,
0,1,0.8
1,1,0.7
2,1,1.5
3,1,0.4
4,1,1.1
...,...,...
10250,99,99.2
10251,99,98.5
10252,99,99.2


In [94]:
LAYOUT_PARAMS = {
    "height": 600,
    "width": 800,
    "font": dict(size=18),
    "xaxis_title": "Time (s)",
    "yaxis_title": "Humidity (%)",
    "legend_title_text": None
}

In [95]:
fig = px.line(
    df, x=df.index, y=["Noisy Data", "Clean Data"], title="Simulated Readings"
)
fig.update_traces(line=dict(width=3), selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), selector={"name": "Noisy Data"})

for threshold in [20, 40, 50, 60, 70]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )

fig.update_layout(dict1=LAYOUT_PARAMS)

display(fig)

In [96]:
x_range = range(1600, 2200)

fig = px.line(
    df.iloc[x_range], x=df.index[x_range], y=["Noisy Data", "Clean Data"], title="Simulated Readings (Zoomed)"
)
fig.update_traces(line=dict(width=3), selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), selector={"name": "Noisy Data"})

fig.update_layout(LAYOUT_PARAMS)

for threshold in [20]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1,
        # annotation_text="",
        # annotation_position="bottom right"
    )

display(fig)

## Implement Filtering Methods

### Rolling Average

- **Pros**
  - Simple and straightforward to implement
  - Changing the window size is intuitive
- **Cons**
  - Weights newer samples the same as older ones
  - Requires holding the previous $n$ samples in memory
    - Changing the window size also changes the number of samples required in memory

In [97]:
df["Filtered Data (Rolling)"] = df["Noisy Data"].rolling(60).mean().round(1)

### Exponential Weighted Mean

$$
\begin{align}
y_0 &= x_0 \\\\
y_t &= (1 - \alpha) y_{t-1} + \alpha x_t 
\end{align}
$$

- **Pros**
  - Weights newer samples more than older samples
  - Only requires holding the current sample and the previous filter output in memory
    - Tuning the weighting parameter does not change the memory requirements
- **Cons**
  - Less intuitive than a rolling average in the sense that the contribution of each sample to the filter output is not immediatley obvious

- Also to note, this is used in TCP implementations to measure network latency
  - The measured packet round trip time (RTT) is EWM-filtered (with a different $\alpha$)

In [98]:
df["Filtered Data (EWM)"] = df["Noisy Data"].ewm(alpha=0.015, adjust=False).mean().round(1)

In [99]:
x_range = range(4600, 5100)

fig = px.line(
    df.iloc[x_range],
    x=df.index[x_range],
    y=["Noisy Data", "Clean Data", "Filtered Data (Rolling)", "Filtered Data (EWM)"],
    title="Simulated Readings (Zoomed)",
)
fig.update_traces(line=dict(width=2), opacity=0.8, selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), opacity=0.8, selector={"name": "Noisy Data"})
fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (Rolling)"},
)
fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (EWM)"},
)

for threshold in [50]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )

fig.update_layout(LAYOUT_PARAMS)

display(fig)

## Implement Measurement to LED Pattern Converter

In [100]:
def render_pattern(
    value: float, thresholds: list[int] = [0, 20, 40, 50, 60, 70]
) -> list[bool]:
    """
    Render a value as a discrete pattern.

    Arguments:
        value: the value to render
        thresholds: the pattern cutoff values

    Returns
        The pattern as a list of boolean states.
    """
    pattern = [None] * len(thresholds)

    for i, threshold in enumerate(thresholds):
        pattern[i] = True if value >= threshold else False

    return pattern

### Test Rendering 

In [101]:
%%ipytest

def test_negative():
    assert render_pattern(-1) == [False] * 6

def test_large_value():
    assert render_pattern(100000) == [True] * 6

def test_threshold_low_boundary():
    assert render_pattern(40) == [True, True, True, False, False, False]

def test_threshold_middle():
    assert render_pattern(45) == [True, True, True, False, False, False]

def test_threshold_high_boundary():
    assert render_pattern(49) == [True, True, True, False, False, False]


.....                                                                                        [100%]
5 passed in 0.01s


### Example

## Convert the Noisy and Filtered Sample Data

- To visualize the results the patterns will be represented as integers
  - [True, True, False, False, False, False] --> 2

In [102]:
# This is just a helper for converting the values
# Not intended as a production prototype

def render_pattern_sum_helper(value: float) -> int:
    return sum(render_pattern(value))

In [103]:
df["Noisy Pattern"] = df["Noisy Data"].apply(render_pattern_sum_helper)
df["Filtered Pattern (Rolling)"] = df["Filtered Data (Rolling)"].apply(render_pattern_sum_helper)
df["Filtered Pattern (EWM)"] = df["Filtered Data (EWM)"].apply(render_pattern_sum_helper)

## Visualize the Patterns

In [104]:
fig = px.line(
    df,
    x=df.index,
    y=[
        "Noisy Pattern",
        "Filtered Pattern (Rolling)",
        "Filtered Pattern (EWM)",
    ],
    title="Simulated Patterns",
)
fig.update_traces(line=dict(width=2), selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), selector={"name": "Noisy Data"})

fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (Rolling)"},
)
fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (EWM)"},
)

fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_yaxes(title="Pattern Number")

display(fig)

In [105]:
fig = px.line(
    df.loc[x_range],
    x=df.index[x_range],
    y=[
        "Noisy Pattern",
        "Filtered Pattern (Rolling)",
        "Filtered Pattern (EWM)",
    ],
    title="Simulated Patterns (Zoomed)",
)

fig.update_traces(line=dict(width=1), opacity=0.5, selector={"name": "Noisy Pattern"})
fig.update_traces(line=dict(width=2), selector={"name": "Filtered Pattern (Rolling)"})
fig.update_traces(line=dict(width=2), selector={"name": "Filtered Pattern (EWM)"})

fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_yaxes(title="Pattern Number", dtick=1)

display(fig)

## Conclusion

- The sensor data should be filtered to avoid rapid oscillations near the pattern boundaries
- The actual filter noise will need to be characterized and then used to tune the production filter
- EWM and rolling average methods are both viable (after tuning), but EWM is more space efficient